In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, roc_curve
import nltk
from nltk.corpus import stopwords
import re
from tabulate import tabulate
import plotly.express as px
import plotly.graph_objects as go

In [2]:
# Load the dataset
df = pd.read_csv('/kaggle/input/real-or-fake-fake-jobposting-prediction/fake_job_postings.csv')

In [3]:
# Initial EDA
print("Dataset Shape:", df.shape)
print("Dataset Info:")
print(df.info())
print("Null Values:")
print(df.isnull().sum())
print("Fraudulent Value Counts:")
print(df['fraudulent'].value_counts())

Dataset Shape: (17880, 18)
Dataset Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 17880 entries, 0 to 17879
Data columns (total 18 columns):
 #   Column               Non-Null Count  Dtype 
---  ------               --------------  ----- 
 0   job_id               17880 non-null  int64 
 1   title                17880 non-null  object
 2   location             17534 non-null  object
 3   department           6333 non-null   object
 4   salary_range         2868 non-null   object
 5   company_profile      14572 non-null  object
 6   description          17879 non-null  object
 7   requirements         15184 non-null  object
 8   benefits             10668 non-null  object
 9   telecommuting        17880 non-null  int64 
 10  has_company_logo     17880 non-null  int64 
 11  has_questions        17880 non-null  int64 
 12  employment_type      14409 non-null  object
 13  required_experience  10830 non-null  object
 14  required_education   9775 non-null   object
 15  industry    

In [4]:
# For text columns, replace missing values with an empty string
text_columns = ['title', 'company_profile', 'description', 'requirements', 'benefits']
df[text_columns] = df[text_columns].fillna(' ')

In [5]:
df['location'].fillna('Unknown', inplace=True)
df['department'].fillna('Unknown', inplace=True)
df['salary_range'].fillna('Not Specified', inplace=True)
df['employment_type'].fillna('Not Specified', inplace=True)
df['required_experience'].fillna('Not Specified', inplace=True)
df['required_education'].fillna('Not Specified', inplace=True)
df['industry'].fillna('Not Specified', inplace=True)
df['function'].fillna('Not Specified', inplace=True)

/tmp/ipykernel_18/1166088697.py:1: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['location'].fillna('Unknown', inplace=True)
/tmp/ipykernel_18/1166088697.py:2: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 

In [6]:
# Text preprocessing function
nltk.download('stopwords')
stop_words = set(stopwords.words('english'))

def preprocess_text(text):
    text = text.lower()
    text = re.sub(r'\d+', '', text)
    text = re.sub(r'\s+', ' ', text)
    text = re.sub(r'[^\w\s]', '', text)
    text = [word for word in text.split() if word not in stop_words]
    return ' '.join(text)

[nltk_data] Downloading package stopwords to /usr/share/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [7]:
# Apply preprocessing to text columns
for col in text_columns:
    df[col] = df[col].apply(preprocess_text)

In [8]:
# Combining text columns into a single feature
df['text'] = df[text_columns].apply(lambda x: ' '.join(x), axis=1)

In [9]:
# Distribution of fraudulent vs non-fraudulent job postings
fig = px.histogram(df, x='fraudulent', title='Distribution of Fraudulent vs Non-Fraudulent Job Postings',
                   labels={'fraudulent': 'Fraudulent'}, color='fraudulent',
                   color_discrete_sequence=['#1f77b4', '#ff7f0e'])
fig.update_layout(
    template='plotly_dark',
    xaxis_title='Fraudulent',
    yaxis_title='Count',
    title_x=0.5,
    font=dict(family="Arial, sans-serif", size=14, color="white"),
    paper_bgcolor='#1e1e1e',
    plot_bgcolor='#1e1e1e',
    xaxis=dict(gridcolor='gray'),
    yaxis=dict(gridcolor='gray')
)
fig.show()

/opt/conda/lib/python3.10/site-packages/plotly/express/_core.py:2065: FutureWarning: When grouping with a length-1 list-like, you will need to pass a length-1 tuple to get_group in a future version of pandas. Pass `(name,)` instead of `name` to silence this warning.
  sf: grouped.get_group(s if len(s) > 1 else s[0])


In [10]:
# Top words in fraudulent job postings
fraudulent_jobs = df[df['fraudulent'] == 1]['text']
non_fraudulent_jobs = df[df['fraudulent'] == 0]['text']

def plot_top_words(text, title):
    word_freq = pd.Series(' '.join(text).split()).value_counts().head(20)
    fig = px.bar(word_freq, x=word_freq.index, y=word_freq.values, title=title,
                 labels={'index': 'Words', 'y': 'Frequency'},
                 color=word_freq.values, color_continuous_scale='Blues')
    fig.update_layout(template='plotly_dark')
    fig.show()

plot_top_words(fraudulent_jobs, 'Top Words in Fraudulent Job Postings')
plot_top_words(non_fraudulent_jobs, 'Top Words in Non-Fraudulent Job Postings')

### Interpretation of Top Words in Job Postings

The graphs below display the most frequent words in fraudulent and non-fraudulent job postings, providing insights into the language used in these job ads.

#### Top Words in Fraudulent Job Postings

- **Work**: The most frequent word, appearing over 1,600 times.
- **Experience**: Commonly mentioned, indicating an emphasis on prior job experience.
- **Skills**: Highlighting the necessity of particular abilities.
- **Amp**: Likely due to formatting issues in the dataset.
- **Team**: Stressing the importance of team dynamics.

Other notable words include **company**, **management**, **business**, **service**, **customer**, **position**, **time**, **engineering**, **data**, **project**, **services**, **new**, **ability**, **years**, and **solutions**.

#### Top Words in Non-Fraudulent Job Postings

- **Work**: Also the most frequent word, appearing over 35,000 times.
- **Experience**: Similarly emphasized, appearing frequently.
- **Team**: Suggesting a focus on collaborative work environments.
- **Business**: Indicative of a professional and corporate setting.
- **Company**: Mentioned frequently, likely referring to the hiring organization.

Other significant words include **new**, **customer**, **skills**, **sales**, **management**, **development**, **working**, **services**, **amp**, **service**, **years**, **people**, **looking**, **marketing**, and **us**.

#### Summary

Both fraudulent and non-fraudulent job postings frequently use terms like **work**, **experience**, **team**, and **skills**. However, non-fraudulent postings have a higher emphasis on business-related terms like **company**, **business**, **customer**, **sales**, and **management**. Fraudulent postings, on the other hand, show a diverse range of terms with a slight focus on **management**, **service**, **position**, and **engineering**.

In [11]:
# Feature extraction using TF-IDF
tfidf = TfidfVectorizer(max_features=5000)
X = tfidf.fit_transform(df['text']).toarray()
y = df['fraudulent']

In [12]:
# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [13]:
# Model training using Multinomial Naive Bayes
model = MultinomialNB()
model.fit(X_train, y_train)

MultinomialNB()

In [14]:
# Predictions
y_pred = model.predict(X_test)
y_pred_prob = model.predict_proba(X_test)[:, 1]

In [15]:
# Evaluation
print("Classification Report:")
print(classification_report(y_test, y_pred))
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))
print("ROC AUC Score:", roc_auc_score(y_test, y_pred_prob))

Classification Report:
              precision    recall  f1-score   support

           0       0.97      1.00      0.98      3395
           1       1.00      0.32      0.49       181

    accuracy                           0.97      3576
   macro avg       0.98      0.66      0.73      3576
weighted avg       0.97      0.97      0.96      3576

Confusion Matrix:
[[3395    0]
 [ 123   58]]
ROC AUC Score: 0.9394576033979121


In [16]:
# Visualization of Confusion Matrix
conf_matrix = confusion_matrix(y_test, y_pred)
fig = px.imshow(conf_matrix, text_auto=True, title='Confusion Matrix')
fig.show()

In [17]:
# ROC Curve
fpr, tpr, thresholds = roc_curve(y_test, y_pred_prob)
roc_fig = go.Figure()
roc_fig.add_trace(go.Scatter(x=fpr, y=tpr, mode='lines', name='ROC Curve', line=dict(color='cyan')))
roc_fig.add_trace(go.Scatter(x=[0, 1], y=[0, 1], mode='lines', name='Random Classifier', line=dict(dash='dash', color='red')))
roc_fig.update_layout(title='ROC Curve', xaxis_title='False Positive Rate', yaxis_title='True Positive Rate', template='plotly_dark')
roc_fig.show()

### ROC Curve Interpretation

The ROC (Receiver Operating Characteristic) curve evaluates the Multinomial Naive Bayes model's performance in classifying fraudulent job postings.

#### Key Points:

- **True Positive Rate (TPR)**: Proportion of actual positives correctly identified.
- **False Positive Rate (FPR)**: Proportion of actual negatives incorrectly identified as positives.
- **Diagonal Line (Orange Dashed)**: Represents random guessing.

#### Interpretation:

- The blue curve above the diagonal indicates better-than-random performance.
- The model shows a good balance between detecting true positives and minimizing false positives.

In [18]:
# Display 10 samples with actual and predicted values
samples = pd.DataFrame({'Actual': y_test, 'Predicted': y_pred})
samples = samples.sample(10).reset_index(drop=True)
print("10 Sample Predictions:")
print(tabulate(samples, headers='keys', tablefmt='fancy_grid'))

10 Sample Predictions:
╒════╤══════════╤═════════════╕
│    │   Actual │   Predicted │
╞════╪══════════╪═════════════╡
│  0 │        0 │           0 │
├────┼──────────┼─────────────┤
│  1 │        0 │           0 │
├────┼──────────┼─────────────┤
│  2 │        0 │           0 │
├────┼──────────┼─────────────┤
│  3 │        0 │           0 │
├────┼──────────┼─────────────┤
│  4 │        0 │           0 │
├────┼──────────┼─────────────┤
│  5 │        0 │           0 │
├────┼──────────┼─────────────┤
│  6 │        0 │           0 │
├────┼──────────┼─────────────┤
│  7 │        0 │           0 │
├────┼──────────┼─────────────┤
│  8 │        0 │           0 │
├────┼──────────┼─────────────┤
│  9 │        0 │           0 │
╘════╧══════════╧═════════════╛
